# Cloud ETL Pipeline — Notebook 1: Pipeline Walkthrough

**Author:** Mike Ichikawa  
**Date:** February 2026

This notebook walks through each stage of the ETL pipeline interactively:
Extract → Transform → Load, with inspection at every step.

Run in **LOCAL_MODE** (no AWS account needed) or with real AWS credentials.

```bash
LOCAL_MODE=true jupyter notebook
```

In [ ]:
import sys, os
sys.path.insert(0, '..')
os.environ.setdefault('LOCAL_MODE', 'true')   # remove this line to use real AWS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from datetime import date, timedelta

import config
from src.extract import extract_all_cities
from src.transform import transform_records
from src.load import S3Loader

plt.rcParams.update({'figure.facecolor': 'white', 'axes.grid': True,
                     'grid.alpha': 0.3, 'font.size': 11})

print(f'LOCAL_MODE: {config.LOCAL_MODE}')
print(f'Cities configured: {len(config.CITIES)}')
print(f'Cities: {[c[0] for c in config.CITIES]}')

---
## Stage 1: Extract

Fetch daily weather summaries for all 10 cities from Open-Meteo.
The archive API returns real historical data — no synthetic data here.

In [ ]:
# Extract for a specific historical date (archive API — always available)
target_date = '2026-01-15'

print(f'Extracting weather data for {target_date}...')
records = extract_all_cities(target_date)
print(f'Successfully fetched: {len(records)}/{len(config.CITIES)} cities')

In [ ]:
# Inspect one raw record
import json
print('Sample raw record (Portland):')
portland = next((r for r in records if r['city'] == 'Portland'), records[0])
print(json.dumps({k: v for k, v in portland.items() if k != 'description'}, indent=2))

In [ ]:
# Quick raw data summary
raw_df = pd.DataFrame(records)
print('Raw extraction summary:')
print(raw_df[['city', 'temperature_2m_mean', 'precipitation_sum',
              'wind_speed_10m_max', 'fetch_ms']].to_string(index=False))

---
## Stage 2: Transform

Clean, enrich, and add derived features: heat index, temp range, wind/precip categories, anomaly flags.

In [ ]:
df = transform_records(records, run_id='nb_walkthrough_001')

print(f'Transform complete: {len(df)} rows, {len(df.columns)} columns')
print(f'\nNew derived columns:')
derived = ['temp_range_c', 'heat_index', 'precipitation_category',
           'wind_category', 'price_tier', 'is_extreme_temp', 'is_heavy_rain', 'is_high_wind']
for col in derived:
    if col in df.columns:
        print(f'  {col}: {df[col].values}')

In [ ]:
# Show enriched data
display_cols = ['city', 'avg_temp_c', 'max_temp_c', 'min_temp_c', 'temp_range_c',
                'heat_index', 'precipitation_mm', 'precipitation_category',
                'wind_speed_max_kmh', 'wind_category', 'is_heavy_rain', 'is_high_wind']
print(df[display_cols].to_string(index=False))

In [ ]:
# Visualize temperature range by city
df_sorted = df.sort_values('avg_temp_c', ascending=False)

fig, ax = plt.subplots(figsize=(12, 6))
fig.suptitle(f'Temperature Summary by City — {target_date}')

x = range(len(df_sorted))
ax.bar(x, df_sorted['temp_range_c'], bottom=df_sorted['min_temp_c'],
       color='#2C7BB6', alpha=0.6, label='Temp range (min→max)', width=0.6)
ax.scatter(x, df_sorted['avg_temp_c'], color='#D7191C', zorder=5, s=80,
           label='Daily mean', marker='D')

ax.set_xticks(x)
ax.set_xticklabels(df_sorted['city'], rotation=30, ha='right')
ax.set_ylabel('Temperature (°C)')
ax.axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
ax.legend()

plt.tight_layout()
plt.savefig('../data/outputs/nb1_temp_by_city.png', dpi=120, bbox_inches='tight')
plt.show()

---
## Stage 3: Load to S3

Upload raw JSON (per city) and processed Parquet to S3.
In LOCAL_MODE this writes to `data/raw/` and `data/processed/`.

In [ ]:
loader = S3Loader()

# Upload raw
raw_keys = loader.upload_raw(records, target_date)
print(f'Raw uploads: {len(raw_keys)}')
for k in raw_keys[:3]:
    print(f'  {k}')
print('  ...')

In [ ]:
# Upload processed Parquet
processed_uri = loader.upload_processed(df, target_date)
print(f'Processed upload: {processed_uri}')

# Verify roundtrip: read it back
df_readback = loader.read_processed(target_date)
print(f'\nReadback shape: {df_readback.shape}')
print(f'Readback dtypes OK: {list(df_readback.dtypes.index)[:5]}')

In [ ]:
# Log the run
loader.log_run({
    'run_id': 'nb_walkthrough_001',
    'date': target_date,
    'cities_extracted': len(records),
    'cities_failed': len(config.CITIES) - len(records),
    'extreme_events': int(df['is_extreme_temp'].sum()),
    'processed_uri': processed_uri,
    'duration_seconds': 4.2,
    'status': 'success',
})
print('Run logged.')

---
## S3 Data Layout Verification

Check the Hive-partitioned key structure.

In [ ]:
# Show the key patterns
test_date = '2026-02-01'
print('S3 key patterns for', test_date)
print(f'  Raw (Portland):  {config.s3_raw_key("portland", test_date)}')
print(f'  Processed:       {config.s3_processed_key(test_date)}')
print(f'  Manifest:        {config.s3_manifest_key(test_date)}')

if config.LOCAL_MODE:
    import os
    print(f'\nLocal processed files:')
    for f in sorted(config.PROCESSED_DIR.glob('*.parquet')):
        size = f.stat().st_size / 1024
        print(f'  {f.name} ({size:.1f} KB)')